In [0]:
# Orders - Data profiling

from pyspark.sql.functions import col, count, when, isnan

orders = spark.read.table("ecommerce.bronze.orders")
orders.printSchema()

# How many nulls per column?
display(orders.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in orders.columns
]))

# Null delivered dates are accurate - Orders that haven't been delivered yet don't have a delivery date. Your transformation needs to preserve these nulls intentionally, not fill them or drop them.

In [0]:
# Customers - Data profiling

customers = spark.read.table("ecommerce.bronze.customers")
customers.printSchema()
customers.select("customer_id", "customer_unique_id").show(10)

# customer_id is a one-time order-scoped identifier. The same physical customer making two purchases gets a different customer_id each time. customer_unique_id is the persistent identifier for the actual person. If you join on customer_id, you will correctly link orders to customers. If you try to count unique customers using customer_id, you'll overcount.

In [0]:
# Order_items - Data profiling

items = spark.read.table("ecommerce.bronze.order_items")
distinct = items.select("order_id").distinct().count()  # unique orders
total = items.count()                                   # total rows
print(f"Distinct orders: {distinct:,}")
print(f"Total rows: {total:,}")

# One order can contain multiple items (e.g., a customer buys three different products). When you join orders to order_items, you will get one row per item per order, not one row per order. This is a fan-out join. You need to decide: is your Silver table at the order grain or the order-item grain?

In [0]:
# Products - Data profiling

products = spark.read.table("ecommerce.bronze.products")
translations = spark.read.table("ecommerce.bronze.product_category_translations")

# How many products have no translation?
products.join(translations, "product_category_name", "left_anti").count()

# Some product categories don't have translations.